# Data Science for Business - Exploring Flight Delays in NYC

*Session 2 · ISLP Lab 2.3 (2.3.4 Graphics, 2.3.7 Loading Data, 2.3.9 Summaries). Joins and seaborn go beyond ISLP.*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
np.random.seed(42)
pd.set_option('display.max_columns', None)  # Show all columns of wide tables

## Problem description

### Flight delays

Flight delays are a common problem in the airline industry. They can be caused by a variety of factors, such as weather, air traffic control, and mechanical issues. Delays can be frustrating for passengers and costly for airlines. In this case study, we will work with a dataset containing flight delay data for all flights departing NYC in 2013. Besides delays, the dataset also includes useful meta-data on airlines, airports, weather, and planes. [More...](https://cran.r-project.org/web/packages/nycflights13/nycflights13.pdf)

The dataset structure is illustrated in the following diagram:

![Dataset structure](https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_02/nycflights13.png)

Data preparation with pandas (filtering, creating variables, joining tables) and plotting with seaborn go beyond what ISLP covers. ISLP Lab 2.3 introduces the basics (loading data, numerical and graphical summaries); for joins and more, see the [pandas user guide on merging](https://pandas.pydata.org/docs/user_guide/merging.html) and the [seaborn tutorial](https://seaborn.pydata.org/tutorial.html).

## Load data

Let's start by loading the `flights` table, taking a look at the first few rows, and displaying the dimensions of the table.

In [ ]:
flights = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_02/flights.csv')

In [ ]:
flights.head(10)

In [ ]:
flights.shape

We continue by loading the remaining tables and taking a look at the first few rows of each table.

In [ ]:
airlines = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_02/airlines.csv')
airlines.head(10)

In [ ]:
airports = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_02/airports.csv')
airports.head(10)

In [ ]:
planes = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_02/planes.csv')
planes.head(10)

In [ ]:
weather = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_02/weather.csv')
weather.head(10)

**Your turn!** Get to know the `planes` and `weather` tables: How many rows and columns do they have? Which columns contain missing values? What is the oldest plane in the data? (Hint: `shape`, `info()`, `isna().sum()`, `describe()`.)

In [ ]:
# YOUR CODE HERE

## Prepare data

Before we start exploring the data, we need to prepare it. This includes cleaning the data (e.g., filtering rows with missing values or outliers), creating new features, and merging tables.

### Filter out observations

Let's check how many values are missing in each column of the `flights` table.

In [ ]:
flights.isna().sum()

About 9,400 flights have no `air_time` (and no `arr_delay`). These are flights that were cancelled or diverted. We exclude them, together with any flights with a `distance` or `air_time` smaller than or equal to 0.

In [ ]:
flights = flights.dropna(subset=['air_time'])
flights = flights.query('distance > 0 and air_time > 0')
flights.shape

### Construct new variables

The way how times are represented in this dataset is quite strange (look at the values `dep_time` or `arr_time` and the separate `hour` and `minute` columns). We will use the timestamps in the `time_hour` column to extract new time variables (`month_name` and `weekday_name`) and then remove the column.

Note that the timestamps are stored in UTC (the trailing `Z`). An evening flight at 9 pm in New York is already on the next day in UTC, so we first convert the timestamps to New York local time.

In [ ]:
flights['time_hour'] = pd.to_datetime(flights['time_hour'], utc=True).dt.tz_convert('America/New_York')
flights[['time_hour', 'month', 'day', 'hour']].head()

In [ ]:
flights['month_name'] = flights['time_hour'].dt.month_name()
flights['weekday_name'] = flights['time_hour'].dt.day_name()
flights = flights.drop(columns=['time_hour'])

In [ ]:
flights.head()

### Join tables

Finally, we will merge some of the tables to create a single table that contains all the information we need for our visualizations. For this, we will use different types of SQL-style joins.

![Types of joins](https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_02/join-venn.png)

We primarily use inner and left joins for matching observations when keys are equal:

![Inner join](https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_02/join-inner.png)

Let's demonstrate the differences between an **inner join** and a **left join** with some examples. A left join keeps every flight, even if it has no match in the right table (the new columns are then missing). An inner join drops flights whose key has no partner. Compare the shapes.

In [ ]:
flights_and_airlines = flights.merge(airlines, on='carrier', how='inner')
flights_and_airlines.shape

In [ ]:
flights_and_airlines = flights.merge(airlines, on='carrier', how='left')
flights_and_airlines.shape

**Your turn!** Do the same with `flights` and `planes` (key: `tailnum`). Do the inner and left join give the same number of rows? Why (not)? What happens to the `year` columns?

In [ ]:
# YOUR CODE HERE

#### Rename columns before joining
Several tables use the same column names for different things: `name` is the airline name in `airlines` and the airport name in `airports`, and `year` is the flight year in `flights` but the build year in `planes`. If we join such tables, pandas adds suffixes like `_x` and `_y`, which are easy to mix up. We therefore give the columns clear names **before** joining.

In [ ]:
airlines_renamed = airlines.rename(columns={'name': 'airline_name'})
airlines_renamed.head()

In [ ]:
planes_renamed = planes.rename(columns={'year': 'plane_year'})
planes_renamed.head()

We need the `airports` table twice: once for the origin airport and once for the destination airport. With `add_prefix`, we create two copies whose columns start with `origin_` and `dest_`, respectively. The key column `faa` is renamed to `origin` and `dest` so that it matches the column name in `flights`.

In [ ]:
origin_airports = airports.add_prefix('origin_').rename(columns={'origin_faa': 'origin'})
origin_airports.head()

In [ ]:
dest_airports = airports.add_prefix('dest_').rename(columns={'dest_faa': 'dest'})
dest_airports.head()

Weather is recorded hourly per airport, so a flight matches only if `origin`, `month`, `day` **and** `hour` agree (a composite key). The columns `year` and `time_hour` are not needed, because `flights` already contains them.

When clocks are set back at the end of daylight saving time (3 November 2013), the hour 1 am occurs twice, so `weather` has two rows for this key at each airport. We keep only the first one, so that every flight matches at most one weather record.

In [ ]:
weather_keys = ['origin', 'month', 'day', 'hour']
weather[weather.duplicated(subset=weather_keys, keep=False)]

In [ ]:
weather_renamed = weather.drop(columns=['year', 'time_hour']).drop_duplicates(subset=weather_keys)
weather_renamed.shape

#### Join one table at a time
Left join of `flights` and the origin airports using `origin` as key. The argument `validate='many_to_one'` makes pandas check that each flight matches at most one airport (otherwise the join would silently duplicate flights).

In [ ]:
flights_and_origin_airports = flights.merge(origin_airports, on='origin', how='left', validate='many_to_one')
flights_and_origin_airports.shape

Left join of `flights` and the destination airports using `dest` as key.

In [ ]:
flights_and_dest_airports = flights.merge(dest_airports, on='dest', how='left', validate='many_to_one')
flights_and_dest_airports.shape

Some destinations (e.g., airports in Puerto Rico) are not listed in `airports`. In a left join, these flights are kept with missing airport information:

In [ ]:
flights_and_dest_airports['dest_name'].isna().sum()

Left join of `flights` and `weather` using the composite key consisting of `origin`, `month`, `day`, and `hour`.

In [ ]:
flights_and_weather = flights.merge(weather_renamed, on=weather_keys, how='left', validate='many_to_one')
flights_and_weather.shape

#### Join all tables
We can now join all the tables (`flights`, `airlines`, `planes`, `airports` twice, `weather`) into one comprehensive table. Thanks to the renaming, no suffixes are needed and every column has a clear meaning.

In [ ]:
flights_all = (
    flights
    .merge(airlines_renamed, on='carrier', how='left', validate='many_to_one')
    .merge(planes_renamed, on='tailnum', how='left', validate='many_to_one')
    .merge(origin_airports, on='origin', how='left', validate='many_to_one')
    .merge(dest_airports, on='dest', how='left', validate='many_to_one')
    .merge(weather_renamed, on=weather_keys, how='left', validate='many_to_one')
)

flights_all.head()

In [ ]:
flights_all.shape

In [ ]:
flights_all.columns

Let's double-check that the joined columns contain what their names promise:

In [ ]:
flights_all[['carrier', 'airline_name', 'origin', 'origin_name', 'dest', 'dest_name', 'year', 'plane_year']].head()

## Explore data through visualizations

We will now visualize categorical and numerical variables to gain insights into the data (compare ISLP Lab 2.3.4 and 2.3.9). We will use the Seaborn library for this purpose. Seaborn is a data visualization library based on Matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics. But it also offers a low-level interface for detailed customization of plots.

Before each plot, ask yourself which question it answers. Rule of thumb: categorical variable → bar chart / count plot; numerical variable → histogram / density / box plot; two numerical variables → scatter plot.

### Plots for categorical variables

We begin by plotting a bar chart of the number of flights by `origin` airport. The function `countplot` is used to show the counts of observations in each categorical bin using bars. Then we use the functions `title`, `xlabel`, and `ylabel` to set a title and labels of the x and y axes, respectively.

In [ ]:
sns.countplot(data=flights_all, x='origin')
plt.title('Number of Flights by Origin Airport')
plt.xlabel('Origin Airport')
plt.ylabel('Number of Flights')
plt.show()

**Your turn!** Do the same for the number of flights by airline.

In [ ]:
# YOUR CODE HERE

To order the bars by frequency, we can use the `order` parameter of the `countplot` function.

In [ ]:
sns.countplot(data=flights_all, x='airline_name', order=flights_all['airline_name'].value_counts().index)
plt.title('Who are the biggest airlines in NYC?')
plt.xlabel('Airline')
plt.ylabel('Number of Flights Leaving NYC')
plt.xticks(rotation=90)
plt.show()

### Plots for numerical variables

Next, we explore the distribution of the `distance` of all flights using the `displot` function. Using the value *hist* for the `kind` parameter, we can plot a histogram. The *bins* parameter specifies the number of bins to use for the histogram.

In [ ]:
sns.displot(data=flights_all, kind='hist', x='distance', bins=30)
plt.title('Distribution of Flight Distances')
plt.xlabel('Distance (miles)')
plt.ylabel('Number of Flights')
plt.show()

**Your turn!** Do the same for `air_time`.

In [ ]:
# YOUR CODE HERE

Using the value *kde* for the `kind` parameter, we can plot a kernel density estimate. The *fill* parameter specifies whether to fill the area under the KDE curve. Note that the y-axis now shows a density (the area under the curve sums to 1), not a number of flights.

In [ ]:
sns.displot(data=flights_all, kind='kde', x='distance', fill=True)
plt.title('Distribution of Flight Distances')
plt.xlabel('Distance (miles)')
plt.ylabel('Density')
plt.show()

If you want to create the same plot for different subsets of the data, you can use the `FacetGrid` function. It maps a low-level function to create a plot (here: `histplot`) to each subset of the data defined by the level of a categorical variable.

In [ ]:
g = sns.FacetGrid(flights_all, col='origin')
g.map(sns.histplot, 'distance')
g.set_axis_labels('Distance (miles)', 'Number of Flights')
plt.show()

Box plots provide another alternative for visualizing the distribution of numerical variables. The low-level function `boxplot` can be used to create a box plot for a numerical variable (here: `distance`), optionally grouped by a categorical variable (here: `origin`).

In [ ]:
sns.boxplot(data=flights_all, y='distance', x='origin')
plt.title('Box Plot of Flight Distances by Origin')
plt.xlabel('Origin Airport')
plt.ylabel('Distance (miles)')
plt.show()

**Your turn!** Do the same for `arr_delay` by airline instead of `distance` by origin airport.

In [ ]:
# YOUR CODE HERE

The above plots show the relationship between a categorical and a numerical variable. We can also visualize the relationship between two numerical variables. The `relplot` function offers various options for this purpose. Here, we use the `scatter` kind to create a scatter plot of `distance` and `air_time`.

With more than 300,000 flights, a scatter plot becomes an unreadable blob (and slow to draw). We therefore plot a random sample of 5,000 flights.

In [ ]:
flights_sample = flights_all.sample(n=5000, random_state=42)

sns.relplot(data=flights_sample, kind='scatter', x='distance', y='air_time', alpha=0.5)
plt.title('Scatter Plot: Distance vs Air Time')
plt.xlabel('Distance (miles)')
plt.ylabel('Air Time (minutes)')
plt.show()

## Your turn!
**Your task:**
1. Create a scatter plot of the departure delay (`dep_delay`) and a numerical weather variable of your choice (e.g., `visib`, `wind_speed`, or `precip`). Use the sample `flights_sample`. Do you see a relationship?
2. Which month has the highest average departure delay? Compute it with `groupby` and visualize it with a bar chart (hint: `sns.barplot`). Use the `month_name` column we created.

In [ ]:
# YOUR CODE HERE